In [1]:
%pip install -U sentence-transformers faiss-cpu

   ---------------------------------------- 0.0/739.8 kB ? eta -:--:--
   ---------------------------------------- 739.8/739.8 kB 6.1 MB/s  0:00:00
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   --- ------------------------------------ 1.3/16.2 MB 7.5 MB/s eta 0:00:02
   -------- ------------------------------- 3.4/16.2 MB 8.8 MB/s eta 0:00:02
   ------------- -------------------------- 5.5/16.2 MB 9.3 MB/s eta 0:00:02
   --------------------- ------------------ 8.7/16.2 MB 10.5 MB/s eta 0:00:01
   ---------------------------- ----------- 11.5/16.2 MB 11.3 MB/s eta 0:00:01
   ------------------------------------ --- 14.9/16.2 MB 12.0 MB/s eta 0:00:01
   ---------------------------------------- 16.2/16.2 MB 12.3 MB/s  0:00:01

   ---------------------------------------- 0/2 [faiss-cpu]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ----------

In [2]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
else:
    BASE_DIR = CWD.parent

PROTOCOLOS_PATH = BASE_DIR / "data" / "raw" / "protocolos_medicos.json"

print(PROTOCOLOS_PATH)
print("Existe:", PROTOCOLOS_PATH.exists())

C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico\data\raw\protocolos_medicos.json
Existe: False


C:\Users\Diogo\AppData\Local\Temp\ipykernel_22308\2639075852.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
# ============================================================
# CÉLULA 3 — Carregar ou recriar protocolos médicos sintéticos
# ============================================================

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

PROTOCOLOS_PATH = DATA_RAW / "protocolos_medicos.json"

if not PROTOCOLOS_PATH.exists():

    print("protocolos_medicos.json não encontrado.")
    print("Recriando protocolos sintéticos...")

    protocolos = [
        {
            "id": "PROTO-001",
            "titulo": "Avaliação inicial de alteração mamária suspeita",
            "conteudo": (
                "Pacientes com alteração mamária suspeita devem ter os achados clínicos "
                "e de imagem revisados pela equipe responsável. Quando houver achados "
                "inconclusivos ou suspeitos, o caso deve ser encaminhado para avaliação "
                "especializada e definição de exames complementares. O assistente de IA "
                "não deve emitir diagnóstico definitivo."
            )
        },
        {
            "id": "PROTO-002",
            "titulo": "Histórico familiar e fatores de risco",
            "conteudo": (
                "Informações sobre histórico familiar de câncer de mama devem ser consideradas "
                "como parte da avaliação clínica. Casos com histórico familiar relevante devem "
                "ser destacados para revisão médica, sem que o sistema determine isoladamente "
                "a presença ou ausência de doença."
            )
        },
        {
            "id": "PROTO-003",
            "titulo": "Exames pendentes",
            "conteudo": (
                "Antes da conclusão da análise, o sistema deve verificar se existem exames "
                "pendentes. Caso existam, deve informar essa condição e recomendar que o "
                "profissional responsável avalie a necessidade de sua realização."
            )
        },
        {
            "id": "PROTO-004",
            "titulo": "Segurança na recomendação clínica",
            "conteudo": (
                "O assistente deve atuar exclusivamente como ferramenta de apoio à decisão. "
                "Não deve prescrever medicamentos, definir doses, emitir diagnóstico definitivo "
                "ou substituir a avaliação do profissional médico responsável."
            )
        },
        {
            "id": "PROTO-005",
            "titulo": "Rastreabilidade da resposta",
            "conteudo": (
                "Toda resposta clínica produzida pelo assistente deve informar quais dados do "
                "paciente foram considerados e quais protocolos institucionais serviram de base "
                "para a resposta."
            )
        }
    ]

    with open(PROTOCOLOS_PATH, "w", encoding="utf-8") as f:
        json.dump(
            protocolos,
            f,
            ensure_ascii=False,
            indent=4
        )

    print("Protocolos recriados com sucesso.")

else:

    with open(PROTOCOLOS_PATH, "r", encoding="utf-8") as f:
        protocolos = json.load(f)

    print("Arquivo encontrado.")

print("Protocolos carregados:", len(protocolos))

for protocolo in protocolos:
    print(protocolo["id"], "-", protocolo["titulo"])

protocolos_medicos.json não encontrado.
Recriando protocolos sintéticos...
Protocolos recriados com sucesso.
Protocolos carregados: 5
PROTO-001 - Avaliação inicial de alteração mamária suspeita
PROTO-002 - Histórico familiar e fatores de risco
PROTO-003 - Exames pendentes
PROTO-004 - Segurança na recomendação clínica
PROTO-005 - Rastreabilidade da resposta


In [5]:
documentos = []

for protocolo in protocolos:

    documentos.append(
        Document(
            page_content=(
                f"Protocolo: {protocolo['id']}\n"
                f"Título: {protocolo['titulo']}\n"
                f"Conteúdo: {protocolo['conteudo']}"
            ),
            metadata={
                "id": protocolo["id"],
                "titulo": protocolo["titulo"]
            }
        )
    )

print("Documentos:", len(documentos))

Documentos: 5


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = FAISS.from_documents(
    documentos,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("RAG criado com sucesso.")

C:\Users\Diogo\anaconda3\envs\fiap_fase3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0912 21:53:24.660000 22308 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
C:\Users\Diogo\anaconda3\envs\fiap_fase3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Diogo\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.

RAG criado com sucesso.


In [7]:
pergunta = """
Paciente apresenta achado mamário suspeito,
histórico familiar positivo e exame complementar pendente.
Quais pontos devem ser considerados?
"""

documentos_encontrados = retriever.invoke(pergunta)

for doc in documentos_encontrados:
    print("=" * 70)
    print("PROTOCOLO:", doc.metadata["id"])
    print("TÍTULO:", doc.metadata["titulo"])
    print(doc.page_content)

PROTOCOLO: PROTO-001
TÍTULO: Avaliação inicial de alteração mamária suspeita
Protocolo: PROTO-001
Título: Avaliação inicial de alteração mamária suspeita
Conteúdo: Pacientes com alteração mamária suspeita devem ter os achados clínicos e de imagem revisados pela equipe responsável. Quando houver achados inconclusivos ou suspeitos, o caso deve ser encaminhado para avaliação especializada e definição de exames complementares. O assistente de IA não deve emitir diagnóstico definitivo.
PROTOCOLO: PROTO-002
TÍTULO: Histórico familiar e fatores de risco
Protocolo: PROTO-002
Título: Histórico familiar e fatores de risco
Conteúdo: Informações sobre histórico familiar de câncer de mama devem ser consideradas como parte da avaliação clínica. Casos com histórico familiar relevante devem ser destacados para revisão médica, sem que o sistema determine isoladamente a presença ou ausência de doença.
